# SafeMaint Qwen3.5-9B Colab Server

This notebook starts the same `ai/qwen_service` FastAPI server used by local Docker. It does not mount Google Drive. Use it for development/demo only; company deployment should run the Qwen service on an internal GPU server. This version exposes the server with ngrok.

In [ ]:
# Runtime configuration
SAFE_MAINT_REPO_URL = ""  # optional: https://github.com/your-org/your-repo.git
PROJECT_ROOT = "/content/safemaint"

QWEN_BASE_MODEL = "Qwen/Qwen3.5-9B"
QWEN_LORA_ADAPTER = "/content/qwen_adapter"
QWEN_API_KEY = "change-this-shared-demo-token"
QWEN_LOAD_IN_4BIT = "true"
QWEN_ANSWER_LOAD_IN_4BIT = "false"
QWEN_MAX_NEW_TOKENS = "768"
QWEN_CLASSIFY_MAX_NEW_TOKENS = "64"

NGROK_AUTH_TOKEN = ""  # required: https://dashboard.ngrok.com/get-started/your-authtoken

# If blank, the notebook will ask you to upload an adapter zip.
ADAPTER_ZIP_URL = ""

In [ ]:
# Install runtime packages. Colab normally already has torch with CUDA.
!pip -q install fastapi uvicorn transformers accelerate peft bitsandbytes safetensors pydantic pyngrok


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

project_root = Path(PROJECT_ROOT)
qwen_service_dir = project_root / "ai" / "qwen_service"
if project_root.exists() and (qwen_service_dir / "main.py").exists():
    print(f"Project already exists: {project_root}")
else:
    if project_root.exists():
        print(f"Removing incomplete project directory: {project_root}")
        shutil.rmtree(project_root)
    if SAFE_MAINT_REPO_URL:
        subprocess.run(["git", "clone", SAFE_MAINT_REPO_URL, str(project_root)], check=True)
    else:
        from google.colab import files
        print("Upload a project zip that contains ai/qwen_service/ or qwen_service/.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No project zip uploaded.")
        archive = Path(next(iter(uploaded.keys()))).resolve()
        unpack_dir = Path("/content/safemaint_upload")
        if unpack_dir.exists():
            shutil.rmtree(unpack_dir)
        shutil.unpack_archive(str(archive), str(unpack_dir))
        # Accept zips created on Windows too. Some zip tools store paths like
        # ai\\qwen_service\\main.py, which Linux treats as flat filenames.
        for path in list(unpack_dir.rglob("*")):
            if "\\" in path.name:
                target = unpack_dir / Path(path.name.replace("\\", "/"))
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(path), str(target))
        candidates = [p for p in unpack_dir.rglob("qwen_service") if (p / "main.py").exists()]
        if not candidates:
            raise RuntimeError("Could not find qwen_service in uploaded zip.")
        candidate = candidates[0]
        if candidate.parent.name == "ai":
            source_root = candidate.parents[1]
            shutil.copytree(source_root, project_root)
        else:
            qwen_service_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(candidate, qwen_service_dir)

if not (qwen_service_dir / "main.py").exists():
    raise RuntimeError(f"qwen_service not found: {qwen_service_dir}")
print(f"Using project root: {project_root}")


In [ ]:
# Prepare LoRA adapter without Google Drive.
from pathlib import Path
import shutil
import subprocess

adapter_dir = Path(QWEN_LORA_ADAPTER)
if adapter_dir.exists() and (adapter_dir / "adapter_config.json").exists():
    print(f"Adapter already exists: {adapter_dir}")
else:
    if adapter_dir.exists():
        shutil.rmtree(adapter_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)
    if ADAPTER_ZIP_URL:
        zip_path = Path("/content/qwen_adapter.zip")
        subprocess.run(["wget", "-O", str(zip_path), ADAPTER_ZIP_URL], check=True)
    else:
        from google.colab import files
        print("Upload the LoRA adapter zip. It must contain adapter_config.json and adapter_model.safetensors.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No adapter zip uploaded.")
        zip_path = Path(next(iter(uploaded.keys()))).resolve()
    unpack_dir = Path("/content/qwen_adapter_unpacked")
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    shutil.unpack_archive(str(zip_path), str(unpack_dir))
    configs = list(unpack_dir.rglob("adapter_config.json"))
    if not configs:
        raise RuntimeError("adapter_config.json not found in adapter zip.")
    found_adapter_dir = configs[0].parent
    for item in found_adapter_dir.iterdir():
        target = adapter_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
print(f"Using adapter dir: {adapter_dir}")


In [ ]:
# Start the Qwen FastAPI server.
import os
import subprocess
import time

os.environ["QWEN_BASE_MODEL"] = QWEN_BASE_MODEL
os.environ["QWEN_LORA_ADAPTER"] = QWEN_LORA_ADAPTER
os.environ["QWEN_API_KEY"] = QWEN_API_KEY
os.environ["QWEN_DEVICE"] = "cuda"
os.environ["QWEN_LOAD_IN_4BIT"] = QWEN_LOAD_IN_4BIT
os.environ["QWEN_ANSWER_DEVICE"] = "cuda"
os.environ["QWEN_ANSWER_LOAD_IN_4BIT"] = QWEN_ANSWER_LOAD_IN_4BIT
os.environ["QWEN_MAX_NEW_TOKENS"] = QWEN_MAX_NEW_TOKENS
os.environ["QWEN_CLASSIFY_MAX_NEW_TOKENS"] = QWEN_CLASSIFY_MAX_NEW_TOKENS
os.environ.setdefault("HF_HOME", "/content/hf_cache")

# NOTE: /v1/classify uses the LoRA-adapter model (4bit). /v1/intent and
# /v1/answer use a SEPARATE model instance (QWEN_ANSWER_LOAD_IN_4BIT, no
# adapter). These are intentionally two separate loaded models, not one
# model with the adapter toggled on/off -- toggling adapters on/off on a
# single 4bit-quantized instance was found to permanently corrupt later
# generations for the rest of the process lifetime (spacing/garbling in the
# output) once /v1/classify had been called at least once. Keep this as two
# instances until that PEFT/bitsandbytes issue is understood, even though it
# costs more VRAM.

subprocess.run("pkill -f 'uvicorn qwen_service.main:app' || true", shell=True, check=False)
subprocess.run("pkill -f 'python.*8020' || true", shell=True, check=False)
subprocess.run("fuser -k 8020/tcp || true", shell=True, check=False)
time.sleep(2)
server_log = open("/content/qwen_server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "qwen_service.main:app", "--host", "127.0.0.1", "--port", "8020"],
    cwd=str(project_root / "ai"),
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

import urllib.request

def wait_for_local_qwen(timeout_seconds=120):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        if server.poll() is not None:
            break
        try:
            with urllib.request.urlopen("http://127.0.0.1:8020/health/live", timeout=5) as response:
                if response.status == 200:
                    return
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    raise RuntimeError(f"Qwen service did not become healthy on 8020: {last_error}")

wait_for_local_qwen()
print("Qwen service process id:", server.pid)

import json

def check_local_qwen_on(base_url="http://127.0.0.1:8020"):
    """Check only the local FastAPI server and schema; do not generate here."""
    status = {
        "server_process_running": server.poll() is None,
        "cuda_requested": os.environ.get("QWEN_DEVICE") == "cuda",
        "health_live": False,
        "openapi_contract_latest": False,
    }
    with urllib.request.urlopen(base_url + "/health/live", timeout=10) as response:
        status["health_live"] = response.status == 200

    with urllib.request.urlopen(base_url + "/openapi.json", timeout=30) as response:
        openapi_payload = json.loads(response.read().decode("utf-8"))
    answer_schema = openapi_payload["components"]["schemas"]["AnswerResponse"]["properties"]
    required_fields = {"answer", "answer_type", "structured_answer", "checklist_items", "used_source_ids", "model"}
    missing = required_fields - set(answer_schema)
    status["openapi_contract_latest"] = not missing
    if missing:
        status["missing_openapi_fields"] = sorted(missing)


    print(json.dumps(status, ensure_ascii=False, indent=2))
    required_ok = status["server_process_running"] and status["health_live"] and status["openapi_contract_latest"]
    if not required_ok:
        raise RuntimeError("Qwen FastAPI server is not ready. Check /content/qwen_server.log.")
    print("Qwen local server is ON. Model load/generation will happen on the first /v1/answer request.")
    return status

qwen_local_status = check_local_qwen_on()
!curl -i http://127.0.0.1:8020/health/live
!curl -s http://127.0.0.1:8020/openapi.json | python -m json.tool | head -n 40
!tail -n 40 /content/qwen_server.log

In [ ]:
# Expose the local FastAPI server with ngrok only.
# This cell creates a fresh random ngrok URL. It does not request a reserved/fixed domain.
import os
import subprocess
import time
from pathlib import Path

import requests
from pyngrok import conf, ngrok

public_url = None
active_tunnel = "ngrok"

# Make sure the local Qwen FastAPI server is alive first. This does not call /v1/answer.
local_health = requests.get("http://127.0.0.1:8020/health/live", timeout=30)
print("Local health status:", local_health.status_code)
print(local_health.text[:500])
local_health.raise_for_status()

# Read the token from the notebook variable first, then Colab secrets/env if available.
ngrok_token = str(globals().get("NGROK_AUTH_TOKEN", "") or os.environ.get("NGROK_AUTH_TOKEN", "")).strip()
try:
    from google.colab import userdata
    ngrok_token = ngrok_token or str(userdata.get("NGROK_AUTH_TOKEN") or "").strip()
except Exception:
    pass

if not ngrok_token:
    raise RuntimeError("NGROK_AUTH_TOKEN is empty. Put your token in the first configuration cell, then rerun cells 1-6.")

# Kill only this Colab runtime's ngrok process and ignore any old/fixed ngrok config.
try:
    for tunnel_info in ngrok.get_tunnels():
        try:
            ngrok.disconnect(tunnel_info.public_url)
        except Exception:
            pass
except Exception:
    pass
ngrok.kill()
subprocess.run("pkill -f ngrok || true", shell=True, check=False)
time.sleep(2)

for env_name in ("NGROK_DOMAIN", "NGROK_HOSTNAME", "NGROK_EDGE", "NGROK_URL", "NGROK_CONFIG", "PYNGROK_CONFIG"):
    os.environ.pop(env_name, None)

# Use a clean config file so pyngrok cannot reuse a reserved domain like vocalize-...ngrok-free.dev.
ngrok_config_path = "/content/ngrok-random.yml"
try:
    Path(ngrok_config_path).unlink()
except FileNotFoundError:
    pass

ngrok_config = conf.PyngrokConfig(config_path=ngrok_config_path)
ngrok.set_auth_token(ngrok_token, pyngrok_config=ngrok_config)

try:
    tunnel = ngrok.connect(addr="127.0.0.1:8020", proto="http", pyngrok_config=ngrok_config)
    public_url = tunnel.public_url
except Exception as exc:
    message = str(exc)
    if "ERR_NGROK_334" in message or "already online" in message:
        raise RuntimeError(
            "ngrok says a fixed/reserved endpoint is already online. This notebook is set to request a random URL, "
            "so restart the Colab runtime and run cells 1-6 again. If the same fixed URL appears, stop that endpoint "
            "in the ngrok dashboard or close the old Colab runtime using it."
        ) from exc
    raise RuntimeError("ngrok tunnel failed. Check NGROK_AUTH_TOKEN and /content/qwen_server.log.") from exc

if public_url.startswith("http://"):
    public_url = public_url.replace("http://", "https://", 1)

print("Public tunnel provider:", active_tunnel)
print("Public URL:", public_url)

# Only health/schema checks here. Do not call /v1/answer in this cell.
headers = {"ngrok-skip-browser-warning": "true"}
time.sleep(3)
health = requests.get(public_url + "/health/live", headers=headers, timeout=60)
print("Public health status:", health.status_code)
print(health.text[:500])
health.raise_for_status()

openapi = requests.get(public_url + "/openapi.json", headers=headers, timeout=60)
print("Public openapi status:", openapi.status_code)
openapi.raise_for_status()
answer_schema = openapi.json()["components"]["schemas"]["AnswerResponse"]["properties"]
required_fields = {"answer", "answer_type", "structured_answer", "checklist_items", "used_source_ids", "model"}
missing = required_fields - set(answer_schema)
assert not missing, f"AnswerResponse schema is missing fields: {sorted(missing)}"

print("\nngrok tunnel and OpenAPI schema are reachable.")
print("Copy this into backend .env:")
print(f"QWEN_SERVICE_URL={public_url}")
print("QWEN_INTENT_CLASSIFY_ENABLED=true")
print("QWEN_ACCIDENT_CLASSIFY_ENABLED=true")


## Backend env values
 
 This cell does not call `/v1/answer`. It only prints the values to copy into `.env` after ngrok is ready.

In [ ]:
# Print backend .env values. This cell intentionally does not call /v1/answer.
print("Set this in each teammate .env:")
print("QWEN_ENABLED=true")
print("QWEN_PROVIDER=colab")
print(f"QWEN_SERVICE_URL={public_url}")
print(f"QWEN_API_KEY={QWEN_API_KEY}")
print("QWEN_TIMEOUT_SECONDS=600")
print("QWEN_ALLOW_COMPANY_CONTEXT=true")
print("QWEN_INTENT_CLASSIFY_ENABLED=true")
print("QWEN_ACCIDENT_CLASSIFY_ENABLED=true")
print("QWEN_LOAD_IN_4BIT=true")
print("QWEN_ANSWER_LOAD_IN_4BIT=false")
print("QWEN_MAX_NEW_TOKENS=768")
print("QWEN_CLASSIFY_MAX_NEW_TOKENS=64")
print("QWEN_SOURCE_EXCERPT_CHARS=180")
print("QWEN_DOCUMENT_SOURCE_LIMIT=2")
print("QWEN_COMPONENT_SOURCE_LIMIT=2")
print("QWEN_MAINTENANCE_SOURCE_LIMIT=5")
print("\nOptional warmup, only if you deliberately want to trigger Qwen generation:")
print("# Run a real /v1/answer request from the backend or your own test cell.")